---
# Section 1: Pre-processing
---

Flow: đọc 4 table

I. Làm sạch dữ liệu:
1. categories
2. products
3. reviews
4. stores

mỗi mục:
- thống kê cơ bản (dimensions, info)
- kiểm tra trùng
- kiểm tra missing
- kiểm tra unique từng thuộc tính (mục đích: tìm rác)

II. Chuẩn hóa dữ liệu:
1. kiểm tra mapping (các trường dữ liệu giữa các bảng có join nhau được không) -> tìm data không mapping được để sau này tạo database lưu ý
2. kiểm tra xem có sp nào thuộc nhiều hơn 1 category k
3. kiểm tra category (xem đã phân loại product đúng theo category chưa) -> dùng các phương pháp để xử lý nếu có dữ liệu sai category (vd: tf-idf)

## Đọc dữ liệu và khám phá sơ bộ

Các bước thực hiện: 
- Đọc 4 bảng dữ liệu: categories, products, reviews, stores.
- Thực hiện thống kê cơ bản, kiểm tra trùng, thiếu và giá trị unique cho từng bảng.
- Kiểm tra khả năng ánh xạ giữa các bảng.
- Kiểm tra lại phân loại category của sản phẩm ở mức đơn giản.

### Đọc dữ liệu

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# Đường dẫn thư mục data (tương đối so với notebook)
data_dir = Path("../data")

categories_path = data_dir / "categories.csv"
products_path = data_dir / "products.csv"
reviews_path = data_dir / "reviews.csv"
stores_path = data_dir / "stores.csv"

print("Đọc dữ liệu từ:")
print(f"- categories: {categories_path}")
print(f"- products:   {products_path}")
print(f"- reviews:    {reviews_path}")
print(f"- stores:     {stores_path}")

df_category = pd.read_csv(categories_path)
df_product = pd.read_csv(products_path)
df_review = pd.read_csv(reviews_path)
df_store = pd.read_csv(stores_path)

Đọc dữ liệu từ:
- categories: ..\data\categories.csv
- products:   ..\data\products.csv
- reviews:    ..\data\reviews.csv
- stores:     ..\data\stores.csv


In [6]:
df_category.head(5)

,category_id,category_name,parent_category,source_category
0,1882,Dien gia dung,NaN,diengiadung
1,1884,Đồ dùng nhà bếp,Dien gia dung,diengiadung
2,1892,Nồi điện các loại,Đồ dùng nhà bếp,diengiadung
3,1893,Nồi cơm điện,Nồi điện các loại,diengiadung
4,1918,Nồi cơm điện tử,Nồi cơm điện,diengiadung


In [7]:
df_product.head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category
0,273634419,1.0,1918,Nồi cơm điện tử Elmich 0.8L RCE-3915 - Hàng Ch...,https://tiki.vn/noi-com-dien-tu-elmich-0-8l-rc...,Elmich,Thông tin sản phẩm ...,1000000,2200000,55,130,5.0,18,diengiadung
1,134550148,1.0,1918,Nồi cơm điện tử Elmich RCE-1789 (1.2 Lít) - Hà...,https://tiki.vn/noi-com-dien-tu-elmich-rce-178...,Elmich,Chất liệu tuyệt đối cho sức khỏeNồi cơm điện t...,1136000,2500000,55,301,4.8,63,diengiadung
2,274106659,1.0,1918,Nồi Cơm Điện Tử Mini Philips HD3170/66 - 600W ...,https://tiki.vn/noi-com-dien-tu-mini-philips-h...,Philips,Nồi cơm điện Dòng 3000 của Philips có dung tíc...,965000,1290000,25,61,4.8,8,diengiadung
3,274965977,1.0,1918,"Nồi cơm điện tử Kangaroo 1.8L model KG18DR12, ...",https://tiki.vn/noi-com-dien-tu-1-8l-model-kg1...,Kangaroo,"Dung tích 1,8LDung tích nồi cơm điện KG18DR12 ...",1210000,2520000,52,16,4.8,5,diengiadung
4,260224713,1.0,1918,Nồi cơm điện cao tần Daewoo DWRC-G411IH 1.8L l...,https://tiki.vn/noi-com-dien-cao-tan-1-8l-daew...,Daewoo,...,1626000,2990000,46,9,3.0,2,diengiadung


In [8]:
df_review.head(5)

,review_id,product_id,user_name,rating,review_text,like_count,review_date,source_category
0,20007421,273634419,Hồ Hữu Nghĩa,5,"Sản phẩm xài rất ok. Nấu nhanh, gọn nhẹ, nhiểu...",0,2024-10-16T03:30:42.000Z,diengiadung
1,20205429,273634419,Le On,5,Cực kì hài lòng,0,2026-01-13T06:29:08.000Z,diengiadung
2,20202180,273634419,Luyen pv,5,Cực kì hài lòng,0,2025-12-30T16:02:30.000Z,diengiadung
3,20191102,273634419,Nguyễn Văn Khang,5,Cực kì hài lòng,0,2025-11-16T08:24:31.000Z,diengiadung
4,20149003,273634419,Hoàng Văn Chương,5,Dùng tốt,0,2025-07-20T07:46:41.000Z,diengiadung


In [9]:
df_store.head(5)

,store_id,store_name,store_rating,follower_count,source_category
0,1,Tiki Trading,4.6754,512930,diengiadung
1,308020,GiaDungNhaThongMinh,4.8750,39,diengiadung
2,29960,CUCKOO Store,4.6289,4894,diengiadung
3,160749,Mishio Kachi Official,4.5030,5396,diengiadung
4,360957,Seka Official Store,4.0000,0,diengiadung


## I. Làm sạch dữ liệu

Cho từng bảng `df_category`, `df_product`, `df_review`, `df_store`:
- Thống kê kích thước, kiểu dữ liệu
- Kiểm tra trùng dòng
- Kiểm tra missing
- Lọc dữ liệu.

### `df_category`

#### 1. Thống kê cơ bản

In [10]:
df_category.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   category_id      497 non-null    int64 
 1   category_name    497 non-null    object
 2   parent_category  492 non-null    object
 3   source_category  497 non-null    object
dtypes: int64(1), object(3)
memory usage: 15.7+ KB


Nhận xét ngắn:
- kiểu dữ liệu ntn
- cần ktra gì

#### 2. Check duplicate

#### 3. Check missing

#### 4. Lọc dữ liệu

***df_product***

In [12]:
df_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55883 entries, 0 to 55882
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        55883 non-null  int64  
 1   store_id          55023 non-null  float64
 2   category_id       55883 non-null  int64  
 3   product_name      55883 non-null  object 
 4   product_url       55883 non-null  object 
 5   brand             55881 non-null  object 
 6   description       55830 non-null  object 
 7   price             55883 non-null  int64  
 8   original_price    55883 non-null  int64  
 9   discount_percent  55883 non-null  int64  
 10  sold_count        55883 non-null  int64  
 11  rating_avg        55883 non-null  float64
 12  review_count      55883 non-null  int64  
 13  source_category   55883 non-null  object 
dtypes: float64(2), int64(7), object(5)
memory usage: 6.0+ MB


## II. Chuẩn hóa dữ liệu

### II.1. Kiểm tra mapping giữa các bảng

- `products.product_id` so với `reviews.product_id`
- `products.store_id` so với `stores.store_id`
- `products.category_id` so với `categories.category_id`
- So sánh `source_category` giữa các bảng (nếu có).

Mục tiêu: tìm ra các bản ghi không mapping được để sau này lưu ý khi tạo database.

In [ ]:
# II.1. Kiểm tra mapping giữa các bảng

def check_mapping_sets(left_name, left_series, right_name, right_series, key_name):
    """In thống kê mapping giữa hai tập ID."""
    left_ids = set(left_series.dropna().unique())
    right_ids = set(right_series.dropna().unique())

    only_left = left_ids - right_ids
    only_right = right_ids - left_ids

    print("-" * 80)
    print(f"Kiểm tra mapping theo khóa '{key_name}': {left_name} ↔ {right_name}")
    print(f"  Số ID trong {left_name}:  {len(left_ids)}")
    print(f"  Số ID trong {right_name}: {len(right_ids)}")
    print(f"  ID chỉ có ở {left_name}:  {len(only_left)}")
    print(f"  ID chỉ có ở {right_name}: {len(only_right)}")

    if len(only_left) > 0:
        print(f"  Ví dụ ID chỉ có ở {left_name} (tối đa 10):", list(sorted(only_left))[:10])
    if len(only_right) > 0:
        print(f"  Ví dụ ID chỉ có ở {right_name} (tối đa 10):", list(sorted(only_right))[:10])
    print()


# 1) products.product_id vs reviews.product_id
if "product_id" in products.columns and "product_id" in reviews.columns:
    check_mapping_sets("products", products["product_id"], "reviews", reviews["product_id"], "product_id")
else:
    print("Không tìm thấy cột 'product_id' trong một trong hai bảng products/reviews.")

# 2) products.store_id vs stores.store_id
if "store_id" in products.columns and "store_id" in stores.columns:
    check_mapping_sets("products", products["store_id"], "stores", stores["store_id"], "store_id")
else:
    print("Không tìm thấy cột 'store_id' trong một trong hai bảng products/stores.")

# 3) products.category_id vs categories.category_id
if "category_id" in products.columns and "category_id" in categories.columns:
    check_mapping_sets("products", products["category_id"], "categories", categories["category_id"], "category_id")
else:
    print("Không tìm thấy cột 'category_id' trong một trong hai bảng products/categories.")

# 4) So sánh source_category giữa các bảng (nếu có)
for name, df in [("categories", categories), ("products", products), ("reviews", reviews), ("stores", stores)]:
    if "source_category" in df.columns:
        print(f"Bảng {name} - các giá trị source_category:", df["source_category"].dropna().unique())

--------------------------------------------------------------------------------
Kiểm tra mapping theo khóa 'product_id': products ↔ reviews
  Số ID trong products:  55883
  Số ID trong reviews: 10797
  ID chỉ có ở products:  45086
  ID chỉ có ở reviews: 0
  Ví dụ ID chỉ có ở products (tối đa 10): [np.int64(109821), np.int64(111518), np.int64(171745), np.int64(252970), np.int64(267644), np.int64(311944), np.int64(359479), np.int64(359729), np.int64(361277), np.int64(362084)]

--------------------------------------------------------------------------------
Kiểm tra mapping theo khóa 'store_id': products ↔ stores
  Số ID trong products:  1443
  Số ID trong stores: 1446
  ID chỉ có ở products:  0
  ID chỉ có ở stores: 3
  Ví dụ ID chỉ có ở stores (tối đa 10): [np.int64(3618), np.int64(8013), np.int64(304271)]

--------------------------------------------------------------------------------
Kiểm tra mapping theo khóa 'category_id': products ↔ categories
  Số ID trong products:  393
  Số ID

### II.2. Kiểm tra product thuộc nhiều hơn 1 category

Mục tiêu: tìm những product (theo `product_id`) đang gắn với từ 2 `category_id` trở lên → có khả năng lỗi gán category hoặc trùng dòng không nhất quán.

In [ ]:
# II.2. Product thuộc nhiều hơn 1 category

if "product_id" not in products.columns or "category_id" not in products.columns:
    print("Không đủ cột 'product_id' và 'category_id' trong bảng products để kiểm tra.")
else:
    tmp = products[["product_id", "category_id"]].drop_duplicates()
    counts = (
        tmp.groupby("product_id")["category_id"]
        .nunique()
        .reset_index(name="category_count")
    )

    multi_cat = counts[counts["category_count"] > 1]
    print("Số product_id thuộc nhiều hơn 1 category:", multi_cat.shape[0])

    if multi_cat.empty:
        print("→ Không tìm thấy product nào thuộc nhiều hơn 1 category.")
    else:
        # Join lại để xem chi tiết category và tên sản phẩm (nếu có)
        detail = multi_cat.merge(tmp, on="product_id", how="left")
        if "product_name" in products.columns:
            detail = detail.merge(
                products[["product_id", "product_name"]].drop_duplicates(),
                on="product_id",
                how="left",
            )
        elif "name" in products.columns:
            detail = detail.merge(
                products[["product_id", "name"]].drop_duplicates(),
                on="product_id",
                how="left",
            )

        print("Ví dụ các product có nhiều category (tối đa 50 dòng):")
        display(detail.sort_values("category_count", ascending=False).head(50))
        print("\n→ Hãy kiểm tra lại các product trên để chuẩn hóa lại category.")

Số product_id thuộc nhiều hơn 1 category: 0
→ Không tìm thấy product nào thuộc nhiều hơn 1 category.


### II.3. Kiểm tra lại category của sản phẩm (dựa trên nội dung text)

Ý tưởng đơn giản:
- Dùng tên sản phẩm (hoặc mô tả) để tách từ khóa
- Gom nhóm theo `category_id` (và/hoặc tên category)
- Đếm tần suất từ xuất hiện trong từng category → xem top từ khóa đặc trưng
- Dựa vào trực quan, phát hiện sản phẩm có tên không phù hợp với category.

(Đây là phiên bản đơn giản hơn tf-idf, dễ chạy và hiểu.)

In [ ]:
import re
from collections import Counter, defaultdict

# II.2. Phân tích từ khóa theo category

# Đoán tên cột chứa tên/mô tả sản phẩm
product_text_col = None
for c in ["product_name", "name", "title", "product_title", "short_description"]:
    if c in products.columns:
        product_text_col = c
        break

if product_text_col is None:
    print("Không tìm được cột text mô tả sản phẩm (ví dụ: product_name, name, title, ...).")
else:
    print(f"Sử dụng cột '{product_text_col}' để phân tích từ khóa theo category.")

    # Đoán tên cột category trong bảng categories (dùng để hiển thị cho dễ đọc)
    cat_name_col = None
    for c in ["category_name", "name", "title"]:
        if c in categories.columns and c != "category_id":
            cat_name_col = c
            break

    # Join products với categories để lấy tên category (nếu có)
    if "category_id" in products.columns and "category_id" in categories.columns:
        if cat_name_col is not None:
            prod_cat = products[[product_text_col, "category_id"]].merge(
                categories[["category_id", cat_name_col]], on="category_id", how="left"
            )
        else:
            prod_cat = products[[product_text_col, "category_id"]].copy()
            prod_cat["category_label"] = prod_cat["category_id"].astype(str)
    else:
        # Không join được, dùng category_id (nếu có)
        if "category_id" in products.columns:
            prod_cat = products[[product_text_col, "category_id"]].copy()
            prod_cat["category_label"] = prod_cat["category_id"].astype(str)
        else:
            prod_cat = None

    if prod_cat is None:
        print("Không có thông tin category trong bảng products, bỏ qua bước này.")
    else:
        if cat_name_col is not None and "category_label" not in prod_cat.columns:
            prod_cat["category_label"] = prod_cat[cat_name_col].fillna(prod_cat["category_id"].astype(str))

        prod_cat = prod_cat.dropna(subset=[product_text_col])

        def tokenize(text: str):
            text = str(text).lower()
            # Giữ lại chữ cái, số và khoảng trắng
            text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", " ", text)
            tokens = [t for t in text.split() if len(t) > 1]
            return tokens

        word_counts_by_cat = defaultdict(Counter)

        for _, row in prod_cat.iterrows():
            cat_label = row["category_label"]
            tokens = tokenize(row[product_text_col])
            word_counts_by_cat[cat_label].update(tokens)

        # In top từ khóa cho từng category để quan sát
        for cat_label, counter in word_counts_by_cat.items():
            print("=" * 80)
            print(f"Category: {cat_label}")
            print("Top 20 từ khóa phổ biến:")
            for word, freq in counter.most_common(20):
                print(f"  {word}: {freq}")

        print("\n→ Dựa vào các từ khóa đặc trưng, bạn có thể lọc ra các sản phẩm có tên không phù hợp với category để kiểm tra và xử lý (sửa category, loại bỏ, v.v.).")

Sử dụng cột 'product_name' để phân tích từ khóa theo category.
Category: Nồi cơm điện tử
Top 20 từ khóa phổ biến:
  nồi: 374
  cơm: 352
  chính: 343
  hãng: 342
  hàng: 339
  điện: 314
  tử: 184
  lít: 136
  cao: 132
  tần: 118
  tiger: 77
  8l: 70
  suất: 48
  lòng: 43
  rc: 40
  hành: 40
  công: 39
  bảo: 39
  dung: 36
  tích: 36
Category: Nồi cơm nắp gài
Top 20 từ khóa phổ biến:
  cơm: 320
  nồi: 316
  chính: 305
  hãng: 305
  hàng: 298
  điện: 283
  lít: 110
  nắp: 97
  gài: 95
  8l: 89
  màu: 57
  nhiên: 52
  ngẫu: 50
  sunhouse: 50
  dung: 36
  tích: 36
  chống: 35
  2l: 34
  lòng: 32
  công: 30
Category: Nồi cơm nắp rời
Top 20 từ khóa phổ biến:
  cơm: 90
  nồi: 88
  chính: 84
  hãng: 84
  hàng: 82
  điện: 64
  rời: 61
  nắp: 60
  lít: 48
  ksh: 35
  sharp: 33
  màu: 14
  ngẫu: 14
  nhiên: 14
  toshiba: 11
  rc: 11
  2l: 7
  mini: 7
  nấu: 6
  8l: 6
Category: Nồi cơm dung tích lớn
Top 20 từ khóa phổ biến:
  cơm: 33
  nồi: 27
  công: 24
  chính: 24
  hãng: 24
  điện: 22
  nghiệp: 

### II.2.b TF-IDF cho từ khóa theo category

Sử dụng `TfidfVectorizer` (sklearn) trên tên/mô tả sản phẩm để:
- Xây dựng ma trận TF-IDF cho toàn bộ sản phẩm
- Tính TF-IDF trung bình của từng từ trong từng `category_label`
- In ra top từ khóa có TF-IDF cao nhất (đặc trưng nhất) cho mỗi category.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# TF-IDF cho từ khóa theo category
if 'prod_cat' not in globals() or prod_cat is None or product_text_col is None:
    print("Chưa có prod_cat hoặc cột text sản phẩm, hãy chạy cell II.2 trước.")
else:
    texts = prod_cat[product_text_col].astype(str).tolist()
    labels = prod_cat['category_label'].astype(str).tolist()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 1),
        max_df=0.9,
        min_df=2
    )

    X = vectorizer.fit_transform(texts)

    # ⚠️ đảm bảo kiểu numpy array chuẩn
    feature_names = np.asarray(vectorizer.get_feature_names_out())

    print("Ma trận TF-IDF:")
    print("- Số sản phẩm:", X.shape[0])
    print("- Số từ khóa (feature):", X.shape[1])

    top_k = 20
    unique_cats = sorted(set(labels))

    for cat in unique_cats:
        idx = [i for i, lbl in enumerate(labels) if lbl == cat]
        if not idx:
            continue

        X_cat = X[idx]

        # ⚠️ fix tương thích numpy 2.x
        mean_tfidf = np.asarray(X_cat.mean(axis=0)).reshape(-1)

        # ⚠️ tránh so sánh array kiểu cũ
        if np.all(mean_tfidf == 0):
            continue

        top_idx = np.argsort(mean_tfidf)[::-1][:top_k]

        print("=" * 80)
        print(f"Category: {cat}")
        print("Top từ khóa theo TF-IDF (trung bình trong category):")

        for i in top_idx:
            print(f"  {feature_names[i]}: {mean_tfidf[i]:.4f}")

    print("\n→ Các từ khóa trên là những từ đặc trưng nhất cho từng category theo TF-IDF.")

Ma trận TF-IDF:
- Số sản phẩm: 55883
- Số từ khóa (feature): 17666
Category: Access Point - Điểm Truy Cập
Top từ khóa theo TF-IDF (trung bình trong category):
  tenda: 0.1369
  wifi: 0.1324
  point: 0.1049
  access: 0.1049
  unifi: 0.1030
  mesh: 0.0865
  phát: 0.0809
  cpe: 0.0645
  ac: 0.0613
  bộ: 0.0588
  gắn: 0.0584
  chính: 0.0570
  hãng: 0.0570
  tốc: 0.0541
  trần: 0.0521
  hàng: 0.0511
  nova: 0.0500
  tp: 0.0497
  độ: 0.0475
  link: 0.0466
Category: Adapter Sạc - Củ Sạc Thường
Top từ khóa theo TF-IDF (trung bình trong category):
  sạc: 0.1852
  củ: 0.1367
  nhanh: 0.1203
  cổng: 0.0935
  usb: 0.0878
  gan: 0.0582
  chính: 0.0574
  hãng: 0.0573
  pd: 0.0566
  20w: 0.0555
  type: 0.0535
  cốc: 0.0516
  hàng: 0.0510
  hoco: 0.0432
  charger: 0.0405
  ugreen: 0.0312
  30w: 0.0308
  điện: 0.0293
  màu: 0.0283
  thoại: 0.0282
Category: Adapter Sạc - Củ Sạc Xe Hơi
Top từ khóa theo TF-IDF (trung bình trong category):
  sạc: 0.1971
  tẩu: 0.1743
  tô: 0.1463
  xe: 0.1353
  hơi: 0.1124

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import numpy as np
from collections import defaultdict

# Phiễu lọc sản phẩm có khả năng sai category dựa trên TF-IDF

# 1. Xác định cột text và chuẩn bị dataframe (bao gồm product_id nếu có)
product_text_col = None
for c in ["product_name", "name", "title", "product_title", "short_description"]:
    if c in products.columns:
        product_text_col = c
        break

if product_text_col is None:
    print("Không tìm được cột text mô tả sản phẩm (ví dụ: product_name, name, title, ...).")
elif "category_id" not in products.columns:
    print("Không có cột category_id trong bảng products, không thể kiểm tra.")
else:
    # Tên cột category trong bảng categories (dùng để hiển thị nhãn dễ đọc nếu cần)
    cat_name_col = None
    for c in ["category_name", "name", "title"]:
        if c in categories.columns and c != "category_id":
            cat_name_col = c
            break

    base_cols = [product_text_col, "category_id"]
    if "product_id" in products.columns:
        base_cols = ["product_id"] + base_cols

    prod_df = products[base_cols].copy()

    if cat_name_col is not None and "category_id" in categories.columns:
        prod_df = prod_df.merge(
            categories[["category_id", cat_name_col]], on="category_id", how="left"
        )
        prod_df["category_label"] = prod_df[cat_name_col].fillna(prod_df["category_id"].astype(str))
    else:
        prod_df["category_label"] = prod_df["category_id"].astype(str)

    prod_df = prod_df.dropna(subset=[product_text_col])

    texts = prod_df[product_text_col].astype(str).tolist()
    labels = prod_df["category_label"].astype(str).tolist()

    # 2. TF-IDF cho toàn bộ sản phẩm
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 1),
        max_df=0.9,
        min_df=2,
    )
    X = vectorizer.fit_transform(texts)

    if X.shape[1] == 0:
        print("Không có feature TF-IDF nào (có thể do dữ liệu quá ít hoặc min_df quá cao).")
    else:
        X_norm = normalize(X, norm="l2", axis=1)

        cats = sorted(prod_df["category_label"].unique().tolist())
        cat_to_idx = {c: i for i, c in enumerate(cats)}

        # 3. Vector đặc trưng cho từng category (trung bình TF-IDF của các sản phẩm trong category đó)
        idx_by_cat = defaultdict(list)
        for i, lbl in enumerate(labels):
            idx_by_cat[lbl].append(i)

        cat_profiles = []
        for cat in cats:
            idxs = idx_by_cat.get(cat, [])
            if not idxs:
                cat_profiles.append(np.zeros(X_norm.shape[1]))
                continue
            X_cat = X_norm[idxs]
            prof = np.asarray(X_cat.mean(axis=0)).reshape(-1)
            norm_val = np.linalg.norm(prof)
            if norm_val > 0:
                prof = prof / norm_val
            cat_profiles.append(prof)

        profiles_mat = np.vstack(cat_profiles).T  # (n_features, n_cats)

        # 4. Tính similarity (cosine) giữa từng sản phẩm và từng category
        S = X_norm @ profiles_mat  # (n_samples, n_cats)
        S = np.asarray(S)

        best_idx = S.argmax(axis=1)
        best_sim = S.max(axis=1)

        actual_idx = np.array([cat_to_idx[lbl] for lbl in labels])

        if S.shape[1] > 1:
            S_sorted = np.sort(S, axis=1)[:, ::-1]
            second_best = S_sorted[:, 1]
        else:
            second_best = np.zeros_like(best_sim)

        margin = best_sim - second_best

        # 5. Mapping từ category_label sang category_id và tên category (dựa trên dữ liệu hiện có)
        label_to_catid = (
            prod_df.groupby("category_label")["category_id"]
            .agg(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else s.iloc[0])
        )
        if cat_name_col is not None:
            label_to_catname = (
                prod_df.groupby("category_label")[cat_name_col]
                .agg(lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else s.iloc[0])
            )
        else:
            label_to_catname = None

        # 6. Phiễu lọc: chọn sản phẩm có predicted category khác actual và confidence đủ cao
        threshold_sim = 0.2
        threshold_margin = 0.05

        same_cat = best_idx == actual_idx
        suspect_mask = (~same_cat) & (best_sim >= threshold_sim) & (margin >= threshold_margin)

        prod_suspect = prod_df.loc[suspect_mask].copy()

        # Thông tin predict
        best_labels = np.array([cats[i] for i in best_idx])
        pred_labels_sus = best_labels[suspect_mask]
        prod_suspect["pred_category_label"] = pred_labels_sus
        prod_suspect["pred_category_id"] = [label_to_catid.get(lbl, np.nan) for lbl in pred_labels_sus]
        if cat_name_col is not None:
            prod_suspect["current_category_name"] = prod_suspect[cat_name_col]
            prod_suspect["pred_category_name"] = [
                label_to_catname.get(lbl, np.nan) for lbl in pred_labels_sus
            ]

        prod_suspect["confidence"] = best_sim[suspect_mask]
        prod_suspect["product_name"] = prod_suspect[product_text_col]

        # 7. Chọn cột output đúng mong muốn
        cols_out = []
        if "product_id" in prod_suspect.columns:
            cols_out.append("product_id")
        cols_out.append("product_name")
        cols_out.append("category_id")
        cols_out.append("pred_category_id")
        if cat_name_col is not None:
            cols_out.append("current_category_name")
            cols_out.append("pred_category_name")
        cols_out.append("confidence")

        print("Số sản phẩm nghi ngờ sai category:", prod_suspect.shape[0])
        display(prod_suspect[cols_out].sort_values("confidence", ascending=False).head(100))
        print("\n→ Bảng trên liệt kê các sản phẩm có khả năng cao đang bị nhầm category (theo TF-IDF).")

Số sản phẩm nghi ngờ sai category: 4010


,product_id,product_name,category_id,pred_category_id,current_category_name,pred_category_name,confidence
29126,273182880,Tân Từ Điển EVEC 666V (Không cảm ứng)-Hàng Chí...,28446,28878,Kim Từ Điển và Các Loại Máy Thông Dịch,Đầu phát karaoke,0.966895
7471,74219216,Máy May Brother FS60X - Hàng Chính Hãng,1998,1999,Máy may điện tử,Máy may cơ,0.957943
37356,197211620,Ốp Lưng Trong Suốt MIPOW SOFT TPU CRYSTAL CLEA...,28574,28590,Bao Da - Ốp Lưng Điện Thoại iPhone,Bao Da - Ốp Lưng Điện Thoại Honor,0.904862
26533,85371196,Loa Rời 2 Tấc B&C SPEAKERS - ITALIA 8FW51 ( 1 ...,8066,11518,Loa Kéo,Loa Bookshelf,0.900880
27915,85371247,Loa Rời 3 tấc B&C SPEAKERS - ITALIA 12CL76 (1 ...,28862,11518,"Loa sân khấu, ngoài trời",Loa Bookshelf,0.900880
...,...,...,...,...,...,...,...
37007,198097807,Ốp Lưng Trong Suốt Dành Cho iPhone 14 Pro Max ...,28574,28590,Bao Da - Ốp Lưng Điện Thoại iPhone,Bao Da - Ốp Lưng Điện Thoại Honor,0.752088
33121,173532409,Popsocket in hình dành cho điện thoại mẫu...,28480,28478,Giá Đỡ - Chân Đế Thường,Móc Dán Đỡ Điện Thoại,0.751943
15612,275752220,Lấy tách giấy trống 2060 : Dùng cho máy photoc...,28944,28940,Drums và Toner Cho Máy In Laser,Linh Kiện và Phụ Kiện Máy In,0.751688
51583,275263646,Bộ Phím Chuột T-Wolf TF100 – Hàng Chính Hãng,1836,28696,Bộ Phím Chuột Văn Phòng,Bộ Phím Chuột Chơi Game,0.750885



→ Bảng trên liệt kê các sản phẩm có khả năng cao đang bị nhầm category (theo TF-IDF).
